**Name:** Maxim Dokukin     
**SID:** 017025164     
**Course:** ISE 201  
**Instructor:** Prof. S. Gupta  
**Date:** November 6, 2025  
**Dataset:** [Daily Exchange Rates per Euro 1999–2022](https://www.kaggle.com/datasets/lsind18/euro-exchange-daily-rates-19992020/versions/59?resource=download)

# Proposal

## 1. Background or context of data selected

I’m pivoting from the “top 5 macro indicators” idea to one manageable dataset. I’ve worked with econ data for the last few years and shipped a CPI/income/housing capstone, but that scope was too big for this class. The euro FX dataset is long, clean, and practical.

*   **Source:** Kaggle dataset by Daria Chemkaeva. Underlying source is the European Central Bank Statistical Data Warehouse (EXR).
    
*   **Collection:** ECB reference rates from major FX venues, one fixing per TARGET business day, around 16:00 CET.
    
*   **Time period:** 1999-01-04 to 2022-01-10.
    
*   **Why this dataset:** daily frequency, multiple currencies, enough history to cover dot-com, 2008, euro-area crisis, COVID.


## 2. Description of the data

*   **File:** `euro-daily-hist_1999_2022.csv`
    
*   **Shape (wide):** ~5,000 dates × ~41 columns (1 date + ~40 currency rate columns).
    
*   **Units:** price of 1 EUR in each foreign currency.
    
*   **Variables (as provided):**
    
    *   `date`
        
    *   currency columns like `USD`, `GBP`, `JPY`, `CHF`, `AUD`, `CAD`, `CNY`, `BGN`, `CZK`, `DKK`, etc. Each is numeric.
        
*   **Meets course variable requirement:** at least two continuous numerical variables are present (any two currency rate columns). For one categorical variable I will reshape to long and use `currency` as categorical.
    
*   **Cases (after reshape to long):** each row becomes a `(date, currency)` observation with a single `rate`.

## 3. Goal and questions I plan to answer

*   How did EUR vs USD/GBP/JPY/CHF/CNY behave across major periods (dot-com, 2008, 2010–12, COVID)?
    
*   Do currencies co-move by region or regime (peg vs float) relative to the euro?
    
*   Did ECB policy windows like March 2015 QE change mean returns or volatility for major pairs?
    
*   Which buckets matter most for basic EUR risk management?

## 4. Data quality checks

*   **Missing values / calendar gaps:** weekends and TARGET holidays. No forward-fill; compute returns within each currency and drop non-finite diffs.
    
*   **Outliers:** z-score by currency; flag if |z|>4. Keep real events, trim obvious glitches.
    
*   **Pegs / near-constants:** Bulgarian lev is fixed at 1.9558 per EUR; label as peg so it doesn’t distort variance-based summaries.
    
*   **Legacy euro currencies:** CYP, EEK, GRD, LTL, LVL, MTL, SIT, SKK. Use data only up to their euro-adoption dates.
    
*   **Duplicates / formats:** enforce unique `(date, currency)` after reshape; parse dates; ensure numeric types.

## 5. References
* [https://www.dataquest.io/blog/free-datasets-for-projects/]
* Kaggle: “Daily Exchange Rates per Euro 1999–2022” (Daria Chemkaeva) [https://www.kaggle.com/datasets/lsind18/euro-exchange-daily-rates-19992020/versions/59?resource=download]

# EDA

## Introduction

I want to study daily EUR exchange rates to understand how major currency pairs behaved through key macro periods and policy regimes.
The dataset is long, clean, and practical for measuring return dynamics, co‑movement, and volatility relevant to EUR risk management

## Data

**Source:** Kaggle dataset by Daria Chemkaeva, underlying European Central Bank Statistical Data Warehouse (EXR).  
Kaggle: “Daily Exchange Rates per Euro 1999–2022” (Daria Chemkaeva)  
Link: https://www.kaggle.com/datasets/lsind18/euro-exchange-daily-rates-19992020/versions/59?resource=download

**Collection:** ECB reference rates from major FX venues. One fixing per TARGET business day, around 16:00 CET.

**Time period:** 1999‑01‑04 to 2022‑01‑10.

**Units:** Columns are the price of **1 EUR** in each foreign currency.

**Unit of observation:** After reshaping from wide to long, each row is a single `(date, currency)` observation with the ECB reference rate for that currency on that date.

**Variables used:** `date`, and currency columns such as `USD, GBP, JPY, CHF, CNY, AUD, CAD, BGN, CZK, DKK, ...`  
For analysis, I focus on major pairs **USD, GBP, JPY, CHF, CNY** plus a broader set for cross‑sectional co‑movement.


## Exploratory Data Analysis

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("/content/euro-daily-hist_1999_2022.csv",
                 na_values = "-")
df = df.rename(columns={"Period\\Unit:": "Date"})
df.columns = (df.columns
              .str.strip('[]')
              .str.lower()
              .str.replace(r'\s+', ' ', regex=True)
              .str.strip())
# df.columns = df.columns.str.lower().str.strip('[]')
df['date'] = pd.to_datetime(df['date'], errors='coerce')

df.head()

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df.dtypes

In [ ]:
df.isna().sum()


In [ ]:
# check duplicates
import pandas as pd
import numpy as np

df_long = (
    df.melt(id_vars='date', var_name='ccy', value_name='rate')
      .dropna(subset=['date', 'ccy'])
      .copy()
)
df_long['ccy'] = df_long['ccy'].str.strip().str.lower()

dupes = df_long.duplicated(['date', 'ccy'])
print("duplicate (date,ccy) rows removed:", int(dupes.sum()))
df_long = df_long[~dupes]

print("rows:", len(df_long), "currencies:", df_long['ccy'].nunique())
print(df_long.dtypes)
df_long.head()

In [ ]:
# calendar gaps and log returns
from dateutil.easter import easter

years = pd.RangeIndex(df_long['date'].dt.year.min(), df_long['date'].dt.year.max() + 1)
holidays = []
for y in years:
    e = easter(int(y))
    holidays.extend([
        pd.Timestamp(y, 1, 1),
        pd.Timestamp(e) - pd.Timedelta(days=2),
        pd.Timestamp(e) + pd.Timedelta(days=1),
        pd.Timestamp(y, 5, 1),
        pd.Timestamp(y, 12, 25),
        pd.Timestamp(y, 12, 26),
    ])
holidays = pd.DatetimeIndex(sorted(set(holidays)))
bday_target = pd.offsets.CustomBusinessDay(holidays=holidays)

def target_gap_count(g):
    if g.empty:
        return 0
    full = pd.date_range(g['date'].min(), g['date'].max(), freq=bday_target)
    return int(pd.Index(full).difference(pd.Index(g['date'])).size)

gaps = (
    df_long.sort_values('date')
    .groupby('ccy')
    .apply(target_gap_count)
    .rename('gap_count')
    .to_frame()
)
print("sample gap counts:")
print(gaps.sort_index().head(10))

df_long = df_long.sort_values(['ccy', 'date'])
df_long['ret'] = np.log(df_long['rate']).groupby(df_long['ccy']).diff()
nonfinite_mask = ~np.isfinite(df_long['ret'])
print("non-finite returns dropped:", int(nonfinite_mask.sum()))
df_ret = df_long[~nonfinite_mask].copy()
df_ret.head()

In [ ]:
# outliers via z-scores
g = df_ret.groupby('ccy')['ret']
mu = g.transform('mean')
sd = g.transform('std').replace(0, np.nan)
df_ret['z'] = (df_ret['ret'] - mu) / sd
df_ret['outlier'] = df_ret['z'].abs() > 4

out_counts = df_ret.groupby('ccy')['outlier'].sum().sort_values(ascending=False)
print("top 10 currencies by outlier count:")
print(out_counts.head(10))
print("share of outliers overall:", float(df_ret['outlier'].mean()))

In [ ]:
# pegs and near-constants
cv = df_ret.groupby('ccy')['rate'].std() / df_ret.groupby('ccy')['rate'].mean()
near_const = (cv.fillna(np.inf) < 5e-4)
peg_set = set(near_const[near_const].index.tolist())
peg_set.add('bulgarian lev')

df_ret['peg'] = df_ret['ccy'].isin(peg_set)
bgn = df_ret[df_ret['ccy'].eq('bulgarian lev')]
bgn_exact_share = float(np.isclose(bgn['rate'], 1.9558, atol=1e-3).mean()) if not bgn.empty else np.nan

print("peg or near-constant currencies:", sorted(peg_set))
print("BGN exact≈1.9558 share:", bgn_exact_share)
print(df_ret[['ccy', 'date', 'rate', 'peg']].head(3))

In [ ]:
# remove post-adoption rows for legacy currencies
adopt = {
    'cypriot pound': '2008-01-01',
    'estonian kroon': '2011-01-01',
    'greek drachma': '2001-01-01',
    'lithuanian litas': '2015-01-01',
    'latvian lats': '2014-01-01',
    'maltese lira': '2008-01-01',
    'slovenian tolar': '2007-01-01',
    'slovak koruna': '2009-01-01',
}
adopt = {k: pd.Timestamp(v) for k, v in adopt.items()}

legacy_mask = df_ret['ccy'].isin(adopt)
kept_mask = ~legacy_mask | (df_ret['date'] < df_ret['ccy'].map(adopt))
removed = int((~kept_mask).sum())
df_final = df_ret[kept_mask].copy()

print("legacy rows removed:", removed)
print("final rows:", len(df_final), "currencies:", df_final['ccy'].nunique())
print(df_final.head(3))

In [ ]:
pd.options.plotting.backend = "matplotlib"
majors = ['us dollar','uk pound sterling','swiss franc']
w_rate = (df_final[df_final['ccy'].isin(majors)]
          .pivot(index='date', columns='ccy', values='rate')
          .sort_index())

ax = w_rate.plot()
ax.set_title('EUR per currency')
ax.set_xlabel('date')
ax.set_ylabel('rate')
ax.legend(title='ccy')
plt.tight_layout()
plt.show()

In [ ]:
pd.options.plotting.backend = "plotly"
fig = df.sort_values('date').set_index('date').plot()
fig.show()
# not too useful

In [ ]:
# list currencies whose max ≤ 2

num = df[cols].apply(pd.to_numeric, errors='coerce')
mx = num.max(skipna=True)
keep = mx[mx <= 3].index.tolist()
print(keep)

long = pd.concat([df['date'], num[keep]], axis=1).melt(id_vars='date', var_name='ccy', value_name='rate').dropna()

import plotly.express as px
fig = px.line(long, x='date', y='rate', color='ccy', title='Currencies with max <= 3')
fig.show()
#  better to split in the bins

In [ ]:
#split all in the bins for more clear output
intervals = [(0, 1), (1, 3), (3, 10), (10, 50), (50, 100), (100, 500), (500, 10000)]

df = df.copy()
df['date'] = pd.to_datetime(df['date'], errors='coerce')
num = df.drop(columns=['date']).apply(pd.to_numeric, errors='coerce')
mx = num.max(skipna=True)

for lo, hi in intervals:
    keep = mx[(mx > lo) & (mx <= hi)].index.tolist()
    print(f"interval {lo}–{hi}:", keep)
    if not keep:
        continue
    long = pd.concat([df['date'], num[keep]], axis=1).melt(id_vars='date', var_name='ccy', value_name='rate').dropna()
    fig = px.line(long, x='date', y='rate', color='ccy', title=f'Currencies with {lo} < max ≤ {hi}')
    fig.show()

## Questions for next stage

- Which currencies lost or gained the most relative to the EUR over the past 20 years?
- How do Russian geopolitical events overlap with EUR/RUB?
- Is there any seasonality in exchange-rate fluctuations?